# Module 8: Capstone — Decision-Memo System

**"Brief in, leadership memo out."**

Combines all four patterns into one complete pipeline.

![Decision-Memo System: Planner + Researcher + Analyzers + Critic → Program Revisor → Decision Memo](./architecture.png)

**Patterns combined:**
- **P2 Parallel heads**: Planner, Researcher, Analyzer A, Analyzer B run simultaneously on the brief
- **P3 Critic-Refiner**: Program Revisor drafts → Critic approves or requests revision → loop
- **P5 Agent-as-Tool**: Orchestrator delegates to each specialist as a callable `@tool`
- **P1 Sequential synthesis**: Program Revisor synthesizes all inputs in sequence

**Agents:** Planner · Researcher · Analyzer (×2) · Program Revisor · Critic

**Prerequisites:** Modules 1–7. This module reuses tools from Module 2.

## Components in This Module

| Component | Pattern | What it does |
|-----------|---------|-------------|
| `parallel_heads` | P2 Fork-Join | Planner + Researcher + Analyzer A + Analyzer B run simultaneously |
| `program_revisor` | P3 Critic-Refiner | Program Revisor drafts → Critic approves or requests revision |
| `orchestrator` | P5 Agent-as-Tool | Coordinates both tools; LLM decides call sequence |

> **The `@tool` docstring IS the routing logic.** The orchestrator reads it to decide when and with what arguments to call each specialist.

In [1]:
%pip install -r requirements.txt


[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
# ── Model configuration ──────────────────────────────────────────────────
# Option 1, Claude Sonnet 4 (default):
#   from strands.models import BedrockModel
#   model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-20250514-v1:0")
# Option 2, Claude Haiku 4.5 (faster, lower cost):
#   model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0")
# Option 3, Amazon Nova Pro (AWS credits):
#   model = BedrockModel(model_id="amazon.nova-pro-v1:0")
print("✅ Setup complete!")

✅ Setup complete!


In [3]:
import sys, os, time, asyncio, json

# decision_brief_tools.py lives in module 02 (it defines the mock data tools used across modules).
# We add that directory to the path here so this notebook can import it directly
# without duplicating the file. In production each specialist has its own copy (see specialists/researcher/).
sys.path.insert(0, os.path.join(os.getcwd(), "..", "02-single-agent"))

import nest_asyncio
nest_asyncio.apply()

from strands import Agent, tool
from strands.multiagent import GraphBuilder
from decision_brief_tools import get_company_data, get_market_benchmarks, get_competitor_data

/Users/eliaws/.pyenv/versions/3.11.7/lib/python3.11/site-packages/pydantic/plugin/_schema_validator.py:39: UserWarning: ImportError while loading the `logfire-plugin` Pydantic plugin, this plugin will not be installed.

ImportError("cannot import name 'ReadableLogRecord' from 'opentelemetry.sdk._logs' (/Users/eliaws/.pyenv/versions/3.11.7/lib/python3.11/site-packages/opentelemetry/sdk/_logs/__init__.py)")
  plugins = get_plugins()


---

## Part 1: System Prompts

All four specialists share one model instance (passed at Agent creation). Narrow, focused prompts: each agent does exactly one job.

In [ ]:
PLANNER_PROMPT = (
    "You are a decision planner. Analyze the brief: identify key questions, "
    "data needed, and criteria for choosing between options. 80 words max."
)

RESEARCHER_PROMPT = (
    "You are a market research specialist. Use tools to gather data. "
    "Return structured findings: data only."
)

FINANCIAL_ANALYZER_PROMPT = (
    "You are a financial analyst. For ALL three options (A, B, C), analyze: "
    "revenue projections, ROI, payback period, budget fit, financial verdict. "
    "Return a structured comparison. 150 words max."
)

RISK_ANALYZER_PROMPT = (
    "You are a risk analyst. For ALL three options (A, B, C), analyze: "
    "implementation complexity (Low/Med/High), top 2 risks per option, mitigations, risk verdict. "
    "Return a structured comparison. 150 words max."
)

PROGRAM_REVISOR_PROMPT = (
    "You are the Program Revisor. You receive: a decision plan, market research, "
    "financial analysis, and risk analysis. "
    "Synthesize them into a COMPLETE leadership decision memo:\n"
    "## Recommendation (one sentence: which option and why)\n"
    "## Options at a Glance (table A/B/C: Complexity, Risk, Financial, Verdict)\n"
    "## Top 3 Risks with specific mitigations\n"
    "## Success Metrics (at least 2 KPIs with numeric targets)\n"
    "## Decision Required (owner, deadline, who approves)\n"
    "Under 400 words. Be direct."
)

CRITIC_GATE_PROMPT = (
    "You are a quality critic. Check ONLY these 5 criteria:\n"
    "1. ## Recommendation with a clear option choice (A, B, or C)\n"
    "2. ## Options at a Glance table comparing A, B, C\n"
    "3. ## Top 3 Risks with at least 3 risks each with a mitigation\n"
    "4. ## Success Metrics with at least 2 KPIs with numeric targets\n"
    "5. ## Decision Required with owner AND deadline\n"
    "Respond: APPROVED or REVISION NEEDED: [criteria numbers missing]"
)

ORCHESTRATOR_PROMPT = (
    "You are the Decision-Memo System orchestrator. Execute:\n"
    "1. Call parallel_heads with the brief — runs all 4 specialists simultaneously.\n"
    "2. Call program_revisor with the brief and parallel_findings — produces the final memo.\n"
    "Execute both steps in order."
)

---

## Part 2: Build the Two Specialist Tools

### Tool 1: `parallel_heads` — 4 specialists in parallel (P2 Fork-Join)
- **Planner**: creates the analysis plan
- **Researcher**: gathers market data with tools
- **Analyzer 1**: financial analysis of all 3 options (ROI, payback, budget fit)
- **Analyzer 2**: risk analysis of all 3 options (complexity, mitigations)

### Tool 2: `program_revisor` — Program Revisor ↔ Critic loop (P3 Critic-Refiner)
Program Revisor synthesizes all 4 parallel findings → Critic evaluates → loop until APPROVED.

In [5]:
@tool
def parallel_heads(brief: str) -> str:
    '''Run 4 specialists simultaneously: Planner, Researcher, Analyzer 1 (financial), Analyzer 2 (risk).

    Each receives the same brief and analyzes it from their perspective.
    Returns combined findings for the Program Revisor.

    Args:
        brief: The full decision brief
    '''
    planner    = Agent(system_prompt=PLANNER_PROMPT,            callback_handler=None)
    researcher = Agent(
        tools=[get_company_data, get_market_benchmarks, get_competitor_data],
        system_prompt=RESEARCHER_PROMPT, callback_handler=None,
    )
    analyzer_1 = Agent(system_prompt=FINANCIAL_ANALYZER_PROMPT, callback_handler=None)
    analyzer_2 = Agent(system_prompt=RISK_ANALYZER_PROMPT,      callback_handler=None)

    async def fork():
        return await asyncio.gather(
            planner.invoke_async(brief),
            researcher.invoke_async(brief),
            analyzer_1.invoke_async(brief),
            analyzer_2.invoke_async(brief),
        )

    plan_out, research_out, fin_out, risk_out = asyncio.run(fork())
    return (
        f"PLAN:\n{plan_out}\n\n"
        f"RESEARCH:\n{research_out}\n\n"
        f"FINANCIAL ANALYSIS:\n{fin_out}\n\n"
        f"RISK ANALYSIS:\n{risk_out}"
    )

In [6]:
CRITIC_GATE_PROMPT = (
    "You are a quality critic. Check ONLY these 5 criteria:\n"
    "1. ## Recommendation with a clear option choice (A, B, or C)\n"
    "2. ## Options at a Glance table comparing A, B, C\n"
    "3. ## Top 3 Risks with at least 3 risks each with a mitigation\n"
    "4. ## Success Metrics with at least 2 KPIs with numeric targets\n"
    "5. ## Decision Required with owner AND deadline\n"
    "Respond: APPROVED or REVISION NEEDED: [criteria numbers missing]"
)


@tool
def program_revisor(brief: str, parallel_findings: str) -> str:
    '''Synthesize parallel findings into a memo, with Critic quality gate.

    Program Revisor drafts from all 4 parallel inputs.
    Critic evaluates: APPROVED or REVISION NEEDED.
    Loop until APPROVED.

    Args:
        brief: The original decision brief
        parallel_findings: Combined output from parallel_heads (plan + research + analyses)
    '''
    revisor = Agent(name="program_revisor", system_prompt=PROGRAM_REVISOR_PROMPT, callback_handler=None)
    critic  = Agent(name="critic",          system_prompt=CRITIC_GATE_PROMPT,     callback_handler=None)

    def needs_revision(state):
        r = state.results.get("critic")
        return bool(r) and "revision needed" in str(r.result).lower()

    builder = GraphBuilder()
    builder.add_node(revisor, "program_revisor")
    builder.add_node(critic,  "critic")
    builder.set_entry_point("program_revisor")
    builder.add_edge("program_revisor", "critic")
    builder.add_edge("critic", "program_revisor", condition=needs_revision)
    builder.set_max_node_executions(6)
    builder.set_execution_timeout(180)
    builder.reset_on_revisit(True)

    result = builder.build()(
        f"Brief:\n{brief}\n\nParallel findings:\n{parallel_findings}"
    )
    for node in reversed(result.execution_order):
        if node.node_id == "program_revisor":
            return str(node.result)
    return str(result)

---

## Part 3: The Orchestrator and Full Brief

In [7]:
DECISION_BRIEF = '''
DECISION BRIEF: NovaCart Premium Tier Launch

Company: NovaCart (2M active users, mid-size e-commerce)
Decision owners: VP Product + CFO approval required

Options:
  Option A: Exclusive Premium: invite-only for top 10% of spenders, $19.99/mo
  Option B: Gradual Rollout: 5% A/B test pilot with kill-switch, $14.99/mo
  Option C: Full Launch: open to all users immediately, $12.99/mo + 30-day free trial

Success target: +15% CLV improvement within 6 months
Budget: $2M  |  Decision deadline: 2027-01-31
'''

orchestrator = Agent(
    tools=[parallel_heads, program_revisor],
    system_prompt=ORCHESTRATOR_PROMPT,
)

print("Running Decision-Memo System...")
print("Step 1: parallel_heads (Planner + Researcher + Analyzers A/B/C)")
print("Step 2: program_revisor (Program Revisor ↔ Critic loop)")
t0 = time.time()
result = orchestrator(DECISION_BRIEF)
elapsed = time.time() - t0
print(f"\nDone in {elapsed:.1f}s")

Running Decision-Memo System...
Step 1: parallel_heads (Planner + Researcher + Analyzers A/B/C)
Step 2: program_revisor (Program Revisor ↔ Critic loop)


I'll execute the full Decision-Memo pipeline

 now — running all four

 specialists in parallel first, then synthesizing into a final reviewed

 memo.

**Step 1 of 2

...

Q2 2024), PrimeStore Unlimited launch data (Q4

 2023). Specialist inputs: Strategic

 Planner · Market Researcher · Financial

 Analyst · Risk Analyst · Program Revisor · Critic.*


Done in 107.4s


---

## Part 4: Inspect the Pipeline

In [8]:
print("=== PIPELINE EXECUTION ===")
call_n = 0
for msg in orchestrator.messages:
    for block in msg.get("content", []):
        if "toolUse" in block:
            call_n += 1
            tu = block["toolUse"]
            inp_keys = list(tu.get("input", {}).keys())
            print(f"  {call_n}. {tu['name']}({', '.join(inp_keys)})")

print()
print("Tool 1 (parallel_heads): Planner + Researcher + Analyzers A/B/C in parallel")
print("Tool 2 (program_revisor): Program Revisor ↔ Critic quality loop")

=== PIPELINE EXECUTION ===
  1. parallel_heads(brief)
  2. program_revisor(brief, parallel_findings)

Tool 1 (parallel_heads): Planner + Researcher + Analyzers A/B/C in parallel
Tool 2 (program_revisor): Program Revisor ↔ Critic quality loop


In [9]:
# Token usage across the full pipeline
summary = result.metrics.get_summary()
usage = summary.get("accumulated_usage", {})

print(f"{'Metric':<25} {'Value':>10}")
print("-" * 37)
print(f"{'Input tokens':<25} {usage.get('inputTokens', 0):>10,}")
print(f"{'Output tokens':<25} {usage.get('outputTokens', 0):>10,}")
print(f"{'Total tokens':<25} {usage.get('totalTokens', 0):>10,}")
print(f"{'Orchestrator cycles':<25} {summary.get('total_cycles', 'n/a'):>10}")
print()

tool_stats = summary.get("tool_usage", {})
if tool_stats:
    print("Per-tool timing:")
    for name, data in tool_stats.items():
        s = data.get("execution_stats", {})
        print(f"  {name}: calls={s.get('call_count',0)} | avg_time={round(s.get('average_time',0),1)}s")

Metric                         Value
-------------------------------------
Input tokens                  12,372
Output tokens                  4,652
Total tokens                  17,024
Orchestrator cycles                3

Per-tool timing:
  parallel_heads: calls=1 | avg_time=30.2s
  program_revisor: calls=1 | avg_time=15.5s


---

## What You Built — All 4 Patterns Combined

| Pattern | Component | Strands API |
|---------|-----------|-------------|
| **P2 Parallel heads** | Planner + Researcher + Analyzer 1 + Analyzer 2 | `asyncio.gather` + `invoke_async` |
| **P3 Critic-Refiner** | Program Revisor ↔ Critic quality loop | `GraphBuilder` + cycle edge |
| **P5 Agent-as-Tool** | Orchestrator delegates via `@tool` functions | `@tool` wrapping sub-pipelines |
| **P1 Sequential** | Parallel phase completes → Program Revisor synthesizes | Implicit sequencing |

**Decision Brief → Leadership Memo:**
- 1 orchestrator (P5) → 2 tools
- 4 parallel specialists (P2): Planner + Researcher + Analyzer×2
- 1 Program Revisor + 1 Critic in quality loop (P3)